# Final Results and Conclusions

**Project:** Predicting 30-Day Hospital Readmission Among Patients with Diabetes  
**Prediction time:** Hospital discharge  
**Final model:** Tuned histogram gradient boosting

This notebook summarizes the completed analysis rather than retraining or choosing a new model. Results come from notebook 05 and saved reports. The top-20% calculation and completed model audits are saved in notebook 05. This report uses the executed outputs, the nine core audit tables, the pairwise significance results, and the expanded fairness/explainability reports. Remaining limitations are identified explicitly.

## 6.1 Evidence and final model

The final cohort contains **69,970 patients**, including **6,277 (8.97%)** with a recorded readmission within 30 days. One encounter per patient was retained using the lowest encounter ID, followed by the specified discharge exclusions and removal of three invalid-gender records.

| Split | Patients | 30-day readmissions |
|---|---:|---:|
| Training | 48,978 | 4,393 |
| Validation | 10,496 | 942 |
| Test | 10,496 | 942 |

The final pipeline includes numeric imputation/scaling, categorical imputation/one-hot encoding, and `HistGradientBoostingClassifier`. Three-fold stratified cross-validation on training data selected **7 maximum leaf nodes** and **L2 regularization = 1.0** from four candidates. Other settings include learning rate 0.1, a maximum of 200 iterations, minimum leaf size 25, early stopping, and random seed 42.

**Evidence files:** `reports/final_gradient_boosting_test_metrics.csv`, `reports/baseline_validation_metrics.csv`, `05_model_evaluation.ipynb`, and `models/final_gradient_boosting.joblib`. The following cell reads the reports and verifies cohort counts without changing files or retraining the model.

In [1]:
from pathlib import Path
import csv

project_dir = Path.cwd()
if not (project_dir / "data/processed/test_fe.csv").exists():
    raise FileNotFoundError("Run this notebook from the AIM_Capstone project folder.")

with (project_dir / "reports/final_gradient_boosting_test_metrics.csv").open() as file:
    final_metrics = next(csv.DictReader(file))

for split in ["train", "val", "test"]:
    with (project_dir / f"data/processed/{split}_fe.csv").open() as file:
        rows = list(csv.DictReader(file))
    positives = sum(int(row["readmit_30"]) for row in rows)
    print(f"{split}: {len(rows):,} patients; {positives:,} readmissions "
          f"({positives / len(rows):.2%})")

print("\nSaved final model:", final_metrics["Model"])
print("Threshold:", final_metrics["Threshold"])
for metric in ["Average precision", "ROC-AUC", "Precision", "Recall", "F1"]:
    print(f"{metric}: {float(final_metrics[metric]):.4f}")
print("Patients flagged:", final_metrics["Patients flagged"])
print(f"Flagged: {float(final_metrics['Flagged (%)']):.2f}%")


train: 48,978 patients; 4,393 readmissions (8.97%)
val: 10,496 patients; 942 readmissions (8.97%)
test: 10,496 patients; 942 readmissions (8.97%)

Saved final model: Tuned gradient boosting
Threshold: 0.1
Average precision: 0.1866
ROC-AUC: 0.6523
Precision: 0.1535
Recall: 0.4777
F1: 0.2323
Patients flagged: 2932
Flagged: 27.93%


In [2]:
from pathlib import Path
import csv

report_dir = Path("reports/final_audit")
audit_reports = {}
for name in ["split_performance", "policy_comparison", "validation_calibration_metrics",
             "validation_calibration_bins", "validation_permutation_importance",
             "subgroup_performance", "test_bootstrap_intervals",
             "training_learning_curve", "patient_identity_checks"]:
    with (report_dir / f"{name}.csv").open() as file:
        audit_reports[name] = list(csv.DictReader(file))
print("Loaded", len(audit_reports), "completed audit reports.")
for name in ["split_performance", "policy_comparison", "test_bootstrap_intervals"]:
    print("\n" + name)
    for row in audit_reports[name]:
        print(row)


Loaded 9 completed audit reports.

split_performance
{'Split': 'train', 'Patients': '48978', 'Positive rate': '0.08969333169994692', 'AP': '0.23750677092148528', 'ROC-AUC': '0.7033937840030708'}
{'Split': 'val', 'Patients': '10496', 'Positive rate': '0.0897484756097561', 'AP': '0.16957922037902073', 'ROC-AUC': '0.6454174661228365'}
{'Split': 'test', 'Patients': '10496', 'Positive rate': '0.0897484756097561', 'AP': '0.1866043776617103', 'ROC-AUC': '0.6523020670969841'}

policy_comparison
{'Split': 'val', 'Policy': 'Threshold 0.10', 'Selected': '2976', 'Selected fraction': '0.28353658536585363', 'Captured': '439', 'Capture': '0.46602972399150744', 'Precision': '0.14751344086021506', 'Lift': '1.6436317147227362'}
{'Split': 'val', 'Policy': 'Top 20%', 'Selected': '2100', 'Selected fraction': '0.20007621951219512', 'Captured': '360', 'Capture': '0.3821656050955414', 'Precision': '0.17142857142857143', 'Lift': '1.9101000909918109'}
{'Split': 'test', 'Policy': 'Threshold 0.10', 'Selected': '2

## 6.2 Model comparison and selection

| Model / representation | Validation average precision | Validation ROC-AUC |
|---|---:|---:|
| Dummy baseline | 0.0897 | 0.5000 |
| Logistic regression — full features | 0.1603 | 0.6378 |
| Logistic regression — 82 selected features | 0.1570 | 0.6347 |
| Logistic regression — 40 SVD components | 0.1435 | 0.6226 |
| Tuned random forest | 0.1648 | 0.6457 |
| Original gradient boosting | 0.1724 | 0.6448 |
| Tuned gradient boosting | 0.1696 | 0.6454 |

All comparisons above use the same validation set. The full logistic representation contained 283 encoded features; selection reduced it to 83 with a small performance decrease. The tested SVD configuration reduced performance further and was not retained.

Original gradient boosting had the highest validation average precision. The tuned version was selected as a tradeoff: its average precision was slightly lower, but its observed training–validation gap was smaller. Random forest had slightly higher validation ROC-AUC than tuned boosting. Therefore, the final model should not be described as the best model on every metric. Paired validation comparisons in notebook 05 use 5,000 stratified bootstrap samples for AP (approximate normal bootstrap-SE p-values), paired DeLong tests for ROC-AUC, and Holm correction across 30 tests. Tuned boosting significantly outperforms SVD logistic regression on AP (see current adjusted p-value table) and ROC-AUC (see current adjusted p-value table), but superiority over full or selected logistic regression, random forest, and original boosting is not established after correction. These are exploratory post-selection comparisons; nonsignificance does not prove equivalence.

In [3]:
import pandas as pd
from IPython.display import display

significance_tests = pd.read_csv("reports/significance_analysis/pairwise_tests.csv")
assert len(significance_tests) == 30
model_contrasts = significance_tests[significance_tests.model_B == "Tuned boosting"].copy()
model_contrasts["Tuned boosting gain"] = -model_contrasts.difference_A_minus_B
display(model_contrasts[["model_A", "metric", "Tuned boosting gain", "p_raw", "p_Holm", "significant_Holm_0.05"]].round(6))
fairness_evidence = pd.read_csv("reports/final_audit/fairness_explainability/reference_contrasts.csv")
display(fairness_evidence[(fairness_evidence.Split == "test") & (fairness_evidence.Policy == "Top 20%")].round(4))


,model_A,metric,Tuned boosting gain,p_raw,p_Holm,significant_Holm_0.05
8,Logistic full,AP,0.009271,0.058611,0.820556,False
9,Logistic full,ROC-AUC,0.007654,0.109882,1.000000,False
16,Logistic selected,AP,0.012571,0.016225,0.308275,False
17,Logistic selected,ROC-AUC,0.010754,0.017299,0.311387,False
22,Logistic SVD,AP,0.026114,0.000217,0.005430,True
23,Logistic SVD,ROC-AUC,0.022842,0.000014,0.000406,True
26,Tuned random forest,AP,0.004789,0.247439,1.000000,False
27,Tuned random forest,ROC-AUC,-0.000283,0.937619,1.000000,False
28,Original boosting,AP,-0.002844,0.364665,1.000000,False
29,Original boosting,ROC-AUC,0.000645,0.785310,1.000000,False


,Split,Policy,Attribute,Group A,Reference,Metric,Group A rate,Reference rate,Difference A minus reference,95% lower,95% upper,Selection ratio A/reference
18,test,Top 20%,race,AfricanAmerican,Caucasian,Selection rate,0.1666,0.2145,-0.0479,-0.0676,-0.0293,0.7765
19,test,Top 20%,race,AfricanAmerican,Caucasian,TPR,0.2771,0.4192,-0.1421,-0.2190,-0.0606,NaN
20,test,Top 20%,race,AfricanAmerican,Caucasian,FPR,0.1563,0.1937,-0.0374,-0.0570,-0.0181,NaN
21,test,Top 20%,gender,Male,Female,Selection rate,0.1882,0.2107,-0.0225,-0.0380,-0.0074,0.8930
22,test,Top 20%,gender,Male,Female,TPR,0.3589,0.4208,-0.0619,-0.1237,0.0024,NaN
23,test,Top 20%,gender,Male,Female,FPR,0.1715,0.1899,-0.0184,-0.0339,-0.0031,NaN


## 6.3 Final test performance

| Metric | Result | Meaning |
|---|---:|---|
| Average precision (AP) | 0.1866 | Better ranking than the constant-probability baseline, whose AP equals test prevalence, approximately 0.0897 |
| ROC-AUC | 0.6523 | Modest ability to rank readmitted patients above non-readmitted patients |
| Decision threshold | 0.10 | Flag scores at or above 0.10 |
| Precision | 15.35% | About one in 6–7 flagged patients was readmitted |
| Recall | 47.77% | Almost half of recorded readmissions were flagged |
| F1 | 0.2323 | Combined summary of precision and recall |
| Patients flagged | 2,932 / 10,496 (27.93%) | Follow-up workload under this threshold |

These are results for the existing fitted model. Average precision is the metric computed by the code; it is not necessarily identical to trapezoidal area under a precision–recall curve. Validation calibration diagnostics are now available in section 6.6. These checks do not establish that every score is an accurate individual clinical risk.

**Operating policy:** The final evaluation uses threshold 0.10 to prioritize recall. Threshold 0.15 gave the highest F1 among the tested validation thresholds, but captured fewer readmissions. The explanation in notebook 05 now matches the evaluated 0.10 policy. Follow-up capacity and error costs still need stakeholder agreement. No policy was changed using test outcomes.

## 6.4 Business objective: highest-risk 20%

This evaluation selects a fixed number of patients by risk rank, rather than applying the 0.10 threshold. The two policies have different workloads.

| Measure | Saved result |
|---|---:|
| Patients selected | 2,100 (20.01%) |
| Readmissions captured | 369 of 942 |
| Capture rate | 39.17% |
| Precision among selected patients | 17.57% |
| Lift relative to random selection | 1.96× |
| Predefined capture target | At least 30% |
| Target achieved | **Yes** |

Selecting the highest-risk fifth of patients captured approximately two-fifths of recorded readmissions, nearly twice the expected capture from random selection. The result exceeds the predefined 30% target. The 20.01% workload reflects rounding upward to a whole patient.

However, **573 readmissions were outside the selected group**, and **1,731 selected patients were not readmitted**. The result supports risk prioritization; it does not show that follow-up would prevent readmissions or establish cost savings.

**Provenance:** Notebook 05 contains the executed ranking calculation. reports/final_audit/policy_comparison.csv records both validation and test policies. Validation top-20% capture is 38.22%, compared with test capture of 39.17%.

## 6.5 Overfitting check

| Gradient-boosting version | Training AP | Validation AP | AP gap |
|---|---:|---:|---:|
| Original | 0.2822 | 0.1724 | 0.1089 |
| Tuned | 0.2375 | 0.1696 | 0.0679 |

Tuning reduced the observed AP gap, but training performance remains higher than validation performance. Some overfitting remains a concern. The tuned model's validation ROC-AUC was 0.6454 and test ROC-AUC was 0.6523; validation AP was 0.1696 and test AP was 0.1866. Test performance did not collapse relative to validation, but this does not prove absence of overfitting or transfer to new hospitals. The approximate test intervals appear below. They use 300 bootstrap resamples, keep the fitted model unchanged, and recalculate the top-20% ranking within each sample. They do not include training uncertainty or hospital clustering.

| Test measure | Estimate | Approximate 95% interval |
| --- | --- | --- |
| AP | 0.1866 | 0.1653–0.2079 |
| ROC-AUC | 0.6523 | 0.6317–0.6710 |
| Threshold precision | 0.1535 | 0.1400–0.1651 |
| Threshold recall | 0.4777 | 0.4430–0.5103 |
| Top-20% capture | 0.3917 | 0.3593–0.4160 |
| Top-20% precision | 0.1757 | 0.1576–0.1898 |
| Top-20% lift | 1.9579 | 1.7959–2.0791 |

## 6.6 Calibration and explainability

### Validation calibration

| Predictor | Brier score | Log loss | Mean score | Observed rate |
| --- | --- | --- | --- | --- |
| Saved gradient boosting | 0.07918 | 0.28917 | 8.92% | 8.97% |
| Training-prevalence baseline | 0.08169 | 0.30196 | 8.97% | 8.97% |

The model has lower Brier score and log loss than the constant training-prevalence baseline. Mean predicted risk is close to the overall observed rate. This supports its probability estimates at an aggregate level, but does not prove perfect calibration.

The validation risk-group table shows local differences. One group has a mean score of 10.51% and observed rate of 8.57%, while the highest-score group has a mean score of 20.20% and observed rate of 21.81%. Inspect the calibration curve alongside these averages. No calibration adjustment was fitted, and individual or external clinical risk accuracy remains unverified.

### Final-model permutation importance

| Feature | Mean validation AP decrease | Repeat standard deviation |
| --- | --- | --- |
| discharge_disposition_id | 0.05616 | 0.00319 |
| number_inpatient | 0.01861 | 0.00176 |
| payer_code | 0.00732 | 0.00287 |
| number_diagnoses | 0.00527 | 0.00139 |
| age_numeric | 0.00250 | 0.00097 |

Discharge disposition has the largest measured AP decrease when shuffled, followed by previous inpatient visits and payer code. I interpret these as variables the fitted model relies on in validation data. The check uses five shuffles per original input column and average precision as the scoring metric.

Correlated features can share information, and shuffling them separately can produce unusual combinations. Repeat standard deviations show shuffle variability, not confidence intervals. These rankings do not establish causes of readmission. Notebook 05 also displays representative validation predictions by outcome group, and includes validation PDP/ICE plots plus representative profile sensitivity. No SHAP analysis was performed; these score responses are not causal explanations.

### Validation PDP and ICE

Notebook 05 inspects score responses over approximately 1,000 validation background profiles, with 60 ICE traces. Mean scenario scores increase from 8.37% at zero previous inpatient visits to 11.71% at one, with the deterministic prior-history features updated together. Other panels inspect age midpoint, hospital stay, laboratory counts and common discharge categories. Representative TP/FP/FN/TN profiles show heterogeneous sensitivity. Correlated features and medically implausible scenarios limit interpretation; these plots describe the fitted model rather than intervention effects.

## 6.7 Data leakage and class imbalance audit

| Safeguard | Evidence / qualification |
|---|---|
| Remove the answer from predictors | `readmitted` is removed after target creation; `readmit_30` is separated from model inputs |
| Remove identifiers from predictors | `patient_nbr` and `encounter_id` are excluded |
| Separate patients | One retained encounter per patient; exact patient/encounter manifest verifies all 69,970 rows, source alignment and zero overlap, with no ambiguous or unmatched records |
| Fit learned preprocessing on training data | Imputation, scaling, encoding, L1 selection, and SVD are in training-fitted pipelines |
| Fit preprocessing inside cross-validation | Grid search receives the complete cloned pipeline |
| Use features available at prediction time | Discharge details and encounter measurements fit the stated discharge-time task; they would not all be available at admission |
| Limit test influence | Visible final evaluation follows model selection; saved code cannot establish the entire historical sequence of test access |

Fixed missing-value labels and predefined diagnosis mappings do not learn statistics from held-out rows. Medication columns with fewer than three training patients recorded as using them are excluded using training data only; the frozen list is applied to every split. Counts and decisions are saved in `reports/final_audit/training_medication_selection.csv`. `data/processed/split_patient_manifest.csv` retains patient/encounter IDs separately from predictors, in split row order. This repairs the earlier full-cohort exclusion rule and ambiguous identity reconstruction; it does not establish every historical test access.

Only 8.97% of the cohort has a positive outcome. Predicting no readmission for everyone would achieve approximately 91.03% accuracy while catching none. Accordingly, AP, ROC-AUC, recall, precision, and workload are more informative than accuracy alone. The workflow uses stratification and threshold comparisons; no resampling or class weighting is shown for the final model. Their absence is not itself a methodological error.

## 6.8 Bias and fairness

Notebook 05 now reports validation and test performance by race, gender, and age under both global policies, including sample sizes, positive counts, selection rates, recall intervals, precision, AP, ROC-AUC, and calibration indicators.

The following table summarizes test results under the **overall top-20% ranking**. Each group does not receive a separate 20% quota. Small-positive groups have fewer than 30 readmissions, a caution flag rather than a guarantee of reliability above that cutoff.

| Attribute | Group | Patients | Positives | Recall | Wilson 95% recall interval | Precision | Small-positive group |
| --- | --- | --- | --- | --- | --- | --- | --- |
| race | AfricanAmerican | 1945 | 166 | 27.71% | 21.46%–34.97% | 14.20% | No |
| race | Asian | 85 | 7 | 42.86% | 15.82%–74.95% | 33.33% | Yes |
| race | Caucasian | 7771 | 718 | 41.92% | 38.37%–45.56% | 18.06% | No |
| race | Hispanic | 215 | 18 | 38.89% | 20.30%–61.38% | 17.95% | Yes |
| race | Other | 169 | 12 | 33.33% | 13.81%–60.94% | 19.05% | Yes |
| race | Unknown | 311 | 21 | 38.10% | 20.75%–59.12% | 20.00% | Yes |
| gender | Female | 5528 | 499 | 42.08% | 37.83%–46.46% | 18.03% | No |
| gender | Male | 4968 | 443 | 35.89% | 31.56%–40.46% | 17.01% | No |
| age | [0-10) | 30 | 1 | 0.00% | 0.00%–79.35% | 0.00% | Yes |
| age | [10-20) | 79 | 2 | 0.00% | 0.00%–65.76% | 0.00% | Yes |
| age | [20-30) | 162 | 13 | 30.77% | 12.68%–57.63% | 28.57% | Yes |
| age | [30-40) | 413 | 31 | 9.68% | 3.35%–24.90% | 9.68% | No |
| age | [40-50) | 1025 | 79 | 22.78% | 14.93%–33.17% | 19.78% | No |
| age | [50-60) | 1811 | 140 | 29.29% | 22.38%–37.30% | 18.06% | No |
| age | [60-70) | 2402 | 214 | 41.59% | 35.19%–48.28% | 20.05% | No |
| age | [70-80) | 2671 | 268 | 47.01% | 41.12%–52.99% | 18.64% | No |
| age | [80-90) | 1645 | 169 | 46.15% | 38.81%–53.67% | 14.29% | No |
| age | [90-100) | 258 | 25 | 40.00% | 23.40%–59.26% | 14.93% | Yes |

AfricanAmerican recall is **27.71%**, compared with **41.92%** for Caucasian patients, an observed difference of 14.21 percentage points. Female recall is **42.08%**, compared with **35.89%** for Male patients. Age-group recall also varies: the 30–40 group captures 9.68% of its 31 positives, while the 70–80 group captures 47.01% of 268 positives.

These are important coverage differences to investigate. They do not by themselves identify the cause, prove discrimination, or certify fairness. Small groups have wide intervals. The intervals are separate within-group uncertainty estimates, not a formal adjusted test across all comparisons.

Next steps should investigate label completeness, differences in recorded utilization and discharge patterns, and group-specific calibration. Any mitigation or comparison of models with and without sensitive attributes should use training/validation data, followed by independent evaluation. No test-derived subgroup threshold was selected.

### Expanded fairness evidence

Notebook 05 now reports demographic parity difference, min/max selection-rate ratio, equal opportunity difference, FPR difference and equalized odds difference. Counts and individual Wilson intervals are retained for all race, gender, age, payer and race × gender groups; payer is an insurance/access proxy, not measured socioeconomic status. Separate summaries flag sparse groups rather than hiding them.

For the fixed test top-20% policy, AfricanAmerican minus Caucasian recall is −14.21 percentage points (individual 2,000-bootstrap 95% interval −21.90 to −6.06); the selection-rate ratio is 0.777. Male minus Female recall is −6.19 points (−12.37 to +0.24); the selection-rate ratio is 0.893. These descriptive intervals are unadjusted and do not certify fairness or identify causes.

The mitigation plan prioritizes outcome completeness, group calibration and data representation, then training-only reweighting or capacity-constrained post-processing with explicit utility/error-rate tradeoffs. Existing test outcomes must not be used to optimize mitigation; independent confirmation requires a new untouched cohort. No mitigation has been implemented or claimed effective.

## 6.9 Error analysis

At the recorded 0.10 threshold, notebook 05 reports this confusion matrix:

| Actual outcome | Not flagged | Flagged |
|---|---:|---:|
| No 30-day readmission | 7,072 true negatives | 2,482 false positives |
| 30-day readmission | 492 false negatives | 450 true positives |

False negatives are patients with a recorded readmission whom the policy would miss. False positives increase follow-up workload among patients without a recorded 30-day readmission. They should not automatically be called unnecessary care: the dataset does not measure whether those patients would benefit from support.

The top-20% policy catches fewer readmissions (369 versus 450) but contacts fewer patients (2,100 versus 2,932), with higher precision (17.57% versus 15.35%). Notebook 05 now summarizes validation error-group counts and median feature values, and displays one example near the median score in each outcome group. These are descriptive checks, not causal explanations. The subgroup audit provides additional evidence about which patients the policies miss.

## 6.10 Limitations, risks, and mitigations

| Limitation | Implication | Proposed mitigation |
|---|---|---|
| Historical 1999–2008 data | Practice and coding may differ from current settings | Validate on a recent independent cohort |
| Lowest encounter ID approximates chronology | Earliest encounter is not confirmed by admission dates | Document the assumption; use timestamps when available |
| One retained encounter per patient | Findings apply to the selected cohort, not necessarily all repeat encounters | Consider all encounters with patient-grouped validation in future work |
| Random historical split | Future-period and new-hospital performance are untested | Use temporal and hospital-level validation when identifiers permit |
| Incomplete capture outside the source network | Some true readmissions may be labelled negative | Seek linked outcome data and disclose this outcome limitation |
| Residual overfitting | Small model differences may be unstable | Bootstrap intervals and learning curves are now available; seek independent confirmation |
| Calibration deviations and subgroup gaps | Probability accuracy and coverage vary | Completed diagnostics identify gaps; investigate and evaluate mitigations using training/validation data |
| Observational associations | No evidence that an intervention reduces readmissions | Evaluate follow-up effectiveness separately |
| Follow-up capacity not established | A fixed threshold may exceed available resources | Notebook 05 now documents the 0.10 policy consistently; agree on capacity with stakeholders |
| Bootstrap intervals condition on the saved model | They omit training uncertainty and hospital clustering | Treat these as approximate internal uncertainty estimates |

## 6.11 Success criteria assessment

| Criterion from the problem statement | Assessment |
|---|---|
| Beat dummy and logistic baselines on validation AP and ROC-AUC | **Met in point estimates only:** tuned boosting AP 0.1696 and ROC-AUC 0.6454 exceed full logistic AP 0.1603 and ROC-AUC 0.6378; statistical superiority over full logistic regression is not established |
| Capture at least 30% of readmissions in the top 20% | **Met in the saved test calculation:** 39.17%, approximate 95% interval 35.93%–41.60% |
| Consistent validation/test performance with no sign of overfitting | **Partially met:** held-out results are broadly similar, but a training–validation gap remains |
| No leakage | **Safeguards implemented, with qualifications:** training-only medication exclusions and exact split identity checks are implemented; historical test access cannot be reconstructed |
| Report race, gender, and age performance | **Reporting completed:** race, gender, and age results are saved; observed gaps need investigation |

## 6.12 Learning curves and final conclusion

### Training-data learning curves

| Training rows per fold | Mean training AP | Mean CV AP | CV AP standard deviation |
| --- | --- | --- | --- |
| 8163 | 0.2445 | 0.1542 | 0.0065 |
| 16326 | 0.2294 | 0.1652 | 0.0012 |
| 32652 | 0.2318 | 0.1792 | 0.0068 |

Cross-validation AP increases from 0.1542 to 0.1792 as training size increases. More training data may help, but training performance remains higher than cross-validation performance. The three folds use training data previously used for tuning, so this is a diagnostic rather than an independent validation result. The saved final model was not replaced.

### Final conclusion

This capstone demonstrates that discharge-time information can provide **modest predictive discrimination** for recorded 30-day readmission in the selected historical diabetes cohort. Tuned gradient boosting improved on the dummy and logistic baselines in validation point estimates, but its superiority over full logistic regression or the leading tree alternatives was not statistically established. Its advantage over SVD logistic regression survives Holm correction on both metrics. On the test set, it achieved AP **0.1866** and ROC-AUC **0.6523**.

The strongest business result is the saved ranking evaluation: selecting **2,100 patients**, approximately the highest-risk **20%**, captured **369 of 942 readmissions (39.17%)**, with **1.96× lift**. This meets the predefined capture target and illustrates how a ranking could prioritize limited follow-up capacity.

The model still misses many readmissions and produces substantial follow-up workload. The completed calibration and subgroup audits provide useful diagnostics, but do not establish intervention benefit, clinically reliable individual risk, fairness, or readiness for deployment. The supported conclusion is that the model offers a useful retrospective prioritization signal, with important limits.

Before submission:

1. Author, course and dataset references are completed in notebook 01. Use the documented notebook order and environment, and inspect the execution verification report before submission.
2. Explain the observed subgroup coverage gaps, the remaining training–validation gap, and the limitations of calibration and uncertainty checks.
3. Agree on the intended follow-up workload before presenting a policy as operationally suitable. The threshold and top-20% policies represent different workloads.

Future work should evaluate a recent external cohort, agree on follow-up capacity and error costs with stakeholders, and measure whether the proposed follow-up actually improves outcomes. Further model or policy selection should use training/validation data rather than repeated optimization on this test set.